In [1]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit, cross_validate, validation_curve
from sklearn.pipeline import Pipeline 
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance

In [2]:
X_train = pd.read_csv("../../03.results/outputs/X_train.csv")
y_train = pd.read_csv("../../03.results/outputs/y_train.csv")
y_reg = y_train["target_next_return"]      # regression target
y_clf = y_train["target_direction"]        # classification target (1 = next day up)
print(X_train.shape, y_train.shape)

(1787, 40) (1787, 4)


In [3]:
tscv = TimeSeriesSplit(n_splits=5)
REG_SCORING = {"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"}
CLF_SCORING = {"accuracy": "accuracy", "f1": "f1", "roc_auc": "roc_auc"}

def cv_scores(model, y, task):
    scoring = REG_SCORING if task == "regression" else CLF_SCORING
    res = cross_validate(model, X_train, y, cv=tscv, scoring=scoring, n_jobs=-1)
    out = {"task": task}
    for k in scoring:
        v = res[f"test_{k}"]
        v = -v if k in ("rmse", "mae") else v
        out[k] = v.mean(); out[k + "_std"] = v.std()
    return out


def plot_curve(model, y, param, values, scoring, title, fname, flip=False):
    tr, va = validation_curve(model, X_train, y, param_name=param, param_range=values, cv=tscv, scoring=scoring, n_jobs=-1)
    s = -1 if flip else 1
    xs = np.arange(len(values)); labels = [str(v) for v in values]
    fig, ax = plt.subplots(figsize=(7, 3.8))
    for arr, name in ((s * tr, "train"), (s * va, "validation")):
        ax.plot(xs, arr.mean(1), marker="o", label=name)
        ax.fill_between(xs, arr.mean(1) - arr.std(1), arr.mean(1) + arr.std(1), alpha=0.15)
    ax.set_xticks(xs); ax.set_xticklabels(labels); ax.set_xlabel(param); ax.set_title(title); ax.legend()
    plt.tight_layout(); fig.savefig("../../03.results/eda_visualizations/" + fname, dpi=130); plt.show()
    return pd.DataFrame({"value": labels, "train": (s * tr).mean(1), "validation": (s * va).mean(1)})

In [ ]:
reg = Pipeline([("model", RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=20, max_features=0.5, n_jobs=1, random_state=RANDOM_STATE))])
clf = Pipeline([("model", RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=20, max_features=0.5, n_jobs=1, random_state=RANDOM_STATE))])
r_reg = cv_scores(reg, y_reg, "regression"); r_clf = cv_scores(clf, y_clf, "classification")
print({k: round(v, 4) for k, v in r_reg.items() if k != "task"}); print({k: round(v, 4) for k, v in r_clf.items() if k != "task"})